# Part B— GNN Notebook 

## Cell 1 — Install dependencies

In [1]:
!pip -q install torch-geometric scikit-image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.3 MB/s eta 0:00:00


## Cell 2 — Imports

In [26]:
import os
import random
import time
import warnings

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision.models as models
import torchvision.transforms as transforms


from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool,
    global_max_pool
)


from sklearn.model_selection import GroupShuffleSplit

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    label_binarize,
    LabelEncoder
)

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    classification_report
)


from imblearn.over_sampling import SMOTE


from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


from skimage.segmentation import slic
from skimage.measure import regionprops
from PIL import Image
from torchvision import transforms


warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


## Cell 3 — Device & seed

In [3]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [4]:
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset"

print("Available datasets under /kaggle/input:")
for name in os.listdir(KAGGLE_INPUT_DIR):
    print(" -", name)

for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    level = root.replace(KAGGLE_INPUT_DIR, "").count(os.sep)
    if level > 2:
        continue
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:
        print(f"{indent}  {f}")

LABELS_PATH = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL_labels.csv"
IMAGE_DIR = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL"

assert os.path.exists(LABELS_PATH), f"Labels CSV not found: {LABELS_PATH}"
assert os.path.isdir(IMAGE_DIR), f"IMAGE_DIR not found: {IMAGE_DIR}"

labels_df = pd.read_csv(LABELS_PATH)
print("Labels shape:", labels_df.shape)
display(labels_df.head())


def find_image_path(file_name, image_dir):
    file_name = str(file_name)
    for ext in ["", ".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        candidate = os.path.join(image_dir, file_name + ext)
        if os.path.exists(candidate):
            return candidate
    for root, _, files in os.walk(image_dir):
        for ext in ["", ".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            target = file_name + ext
            if target in files:
                return os.path.join(root, target)
    return None


image_paths = []
missing_files = []
for fname in tqdm(labels_df["file_name"], desc="Finding images"):
    img_path = find_image_path(fname, IMAGE_DIR)
    if img_path is None:
        missing_files.append(fname)
        image_paths.append(None)
    else:
        image_paths.append(img_path)

df = labels_df.copy()
df["image_path"] = image_paths

if missing_files:
    print(f"\nWARNING: {len(missing_files)} images were not found.")
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

print("Final dataframe shape:", df.shape)
print("Missing images:", len(missing_files))
display(df.head())

Available datasets under /kaggle/input:
 - OCTDL
octdl-optical-coherence-tomography-dataset/
  OCTDL/
    OCTDL_labels.csv
    OCTDL/
Labels shape: (2064, 10)


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321


Finding images: 100%|██████████| 2064/2064 [00:35<00:00, 57.36it/s] 

Final dataframe shape: (2064, 11)
Missing images: 0


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight,image_path
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410,/kaggle/input/datasets/orvile/octdl-optical-co...
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265,/kaggle/input/datasets/orvile/octdl-optical-co...
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410,/kaggle/input/datasets/orvile/octdl-optical-co...
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321,/kaggle/input/datasets/orvile/octdl-optical-co...
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321,/kaggle/input/datasets/orvile/octdl-optical-co...


## Cell 4 — Pretrained ResNet (feature extractor)

In [5]:
weights = models.ResNet18_Weights.IMAGENET1K_V1

resnet = models.resnet18(
    weights=weights
)

resnet.fc = nn.Identity()

resnet = resnet.to(device)
resnet.eval()

for param in resnet.parameters():
    param.requires_grad = False

print("Pretrained ResNet-18 loaded.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


Pretrained ResNet-18 loaded.


## Cell 4.5 — Dataset split (train/val/test)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

candidate_label_cols = [c for c in ["disease", "class", "label", "diagnosis"] if c in df.columns]
LABEL_COL = candidate_label_cols[0] if candidate_label_cols else df.columns[1]
print("Using label column:", LABEL_COL)
print("Available columns:", list(df.columns))

le = LabelEncoder()
df["label_enc"] = le.fit_transform(df[LABEL_COL])
print("Classes:", dict(zip(le.classes_, range(len(le.classes_)))))

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_enc"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_enc"], random_state=SEED
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

Using label column: disease
Available columns: ['file_name', 'disease', 'subcategory', 'condition', 'patient_id', 'eye', 'sex', 'year', 'image_width', 'image_hight', 'image_path']
Classes: {'AMD': 0, 'DME': 1, 'ERM': 2, 'NO': 3, 'RAO': 4, 'RVO': 5, 'VID': 6}
Train: (1444, 12) Val: (310, 12) Test: (310, 12)


## Cell 5 — Edge weight function


In [7]:
def calculate_edge_weight(
    centroid1,
    centroid2
):
    distance = np.linalg.norm(
        np.array(centroid1) -
        np.array(centroid2)
    )

    return 1.0 / (1.0 + distance)

## Cell 6 — Superpixel segmentation + graph builder function


In [8]:
N_SEGMENTS = 75         
CROP_SIZE = 32            

resnet_preprocess = transforms.Compose([
    transforms.Resize((CROP_SIZE, CROP_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def image_to_graph(image_path, label, n_segments=N_SEGMENTS):
    pil_img = Image.open(image_path).convert("RGB")
    img_np = np.array(pil_img)

    segments = slic(
        img_np,
        n_segments=n_segments,
        compactness=10,
        start_label=0
    )

    props = regionprops(segments + 1)  # regionprops needs labels >= 1
    n_nodes = len(props)

    # --- centroids ---
    centroids = [p.centroid for p in props]  # (row, col) per superpixel

    # --- adjacency: pairs of superpixels that touch each other ---
    adjacency_pairs = set()
    # horizontal neighbors
    left, right = segments[:, :-1], segments[:, 1:]
    diff_mask = left != right
    for u, v in zip(left[diff_mask], right[diff_mask]):
        adjacency_pairs.add((min(u, v), max(u, v)))
    # vertical neighbors
    top, bottom = segments[:-1, :], segments[1:, :]
    diff_mask = top != bottom
    for u, v in zip(top[diff_mask], bottom[diff_mask]):
        adjacency_pairs.add((min(u, v), max(u, v)))
    adjacency_pairs = list(adjacency_pairs)

    # --- node features: ResNet-18 embedding of each superpixel's crop ---
    crops = []
    for p in props:
        min_row, min_col, max_row, max_col = p.bbox
        crop = pil_img.crop((min_col, min_row, max_col, max_row))
        crops.append(resnet_preprocess(crop))

    with torch.no_grad():
        batch = torch.stack(crops).to(device)
        node_features = resnet(batch).cpu().numpy()  # (n_nodes, 512)

    # --- edge_index / edge_weight (bidirectional, as in the original Cell 6) ---
    edge_index_list = []
    edge_weight_list = []

    for u, v in adjacency_pairs:
        w = calculate_edge_weight(centroids[u], centroids[v])
        edge_index_list.append([u, v])
        edge_index_list.append([v, u])
        edge_weight_list.append(w)
        edge_weight_list.append(w)

    if len(edge_index_list) == 0:
        # fallback: no touching pairs found (shouldn't normally happen) -> self loops
        edge_index_list = [[i, i] for i in range(n_nodes)]
        edge_weight_list = [1.0 for _ in range(n_nodes)]

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(edge_weight_list, dtype=torch.float32)

    graph = Data(
        x=torch.tensor(node_features, dtype=torch.float32),
        edge_index=edge_index,
        edge_weight=edge_weight,
        y=torch.tensor([label], dtype=torch.long)
    )

    return graph

## Cell 7 — Build graphs for train/val/test splits

In [9]:
def build_graph_list(split_df, desc=""):
    graphs = []
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=desc):
        try:
            g = image_to_graph(row["image_path"], row["label_enc"])
            graphs.append(g)
        except Exception as e:
            print(f"Skipping {row.get('file_name', row.get('image_path'))}: {e}")
    return graphs


train_graphs = build_graph_list(train_df, desc="Train graphs")
val_graphs = build_graph_list(val_df, desc="Val graphs")
test_graphs = build_graph_list(test_df, desc="Test graphs")

y_tr = np.array([g.y.item() for g in train_graphs])

print("Train graphs:", len(train_graphs))
print("Val graphs:", len(val_graphs))
print("Test graphs:", len(test_graphs))
print(train_graphs[0])

Test graphs: 100%|██████████| 310/310 [01:57<00:00,  2.63it/s]

Train graphs: 1444
Val graphs: 310
Test graphs: 310
Data(x=[60, 512], edge_index=[2, 292], y=[1], edge_weight=[292])


## Cell 8 — Weighted GCN model 

In [10]:
class WeightedGCN(nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes,
        dropout=0.3
    ):
        super(WeightedGCN, self).__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = GCNConv(
            hidden_channels,
            hidden_channels
        )

        self.conv3 = GCNConv(
            hidden_channels,
            hidden_channels
        )

        self.dropout = dropout

        self.classifier = nn.Sequential(
            nn.Linear(
                hidden_channels * 2,
                128
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index
        edge_weight = data.edge_weight
        batch = data.batch

        x = self.conv1(
            x,
            edge_index,
            edge_weight
        )

        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index,
            edge_weight
        )

        x = F.relu(x)

        x = self.conv3(
            x,
            edge_index,
            edge_weight
        )

        x = F.relu(x)

        mean_pool = global_mean_pool(
            x,
            batch
        )

        max_pool = global_max_pool(
            x,
            batch
        )

        x = torch.cat(
            [mean_pool, max_pool],
            dim=1
        )

        return self.classifier(x)


## Cell 9 — Class weights (imbalance handling)

In [11]:
classes = np.unique(y_tr)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_tr
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(class_weights)

tensor([ 0.2396,  2.0028,  1.8925,  0.8892, 13.7524,  2.9054,  3.8922],
       device='cuda:0')


## Cell 10 — Model, loss, optimizer

In [12]:
num_features = train_graphs[0].x.shape[1]
num_classes = len(np.unique(y_tr))

model = WeightedGCN(
    in_channels=num_features,
    hidden_channels=128,
    num_classes=num_classes,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print(model)

WeightedGCN(
  (conv1): GCNConv(512, 128)
  (conv2): GCNConv(128, 128)
  (conv3): GCNConv(128, 128)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=7, bias=True)
  )
)


## Cell 11 — DataLoaders



In [13]:
train_loader = DataLoader(
    train_graphs,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=16,
    shuffle=False
)

## Cell 12 — Training function (one epoch)

In [14]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0

    for data in loader:

        data = data.to(device)

        optimizer.zero_grad()

        output = model(data)

        loss = criterion(
            output,
            data.y
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() *
            data.num_graphs
        )

    return total_loss / len(loader.dataset)

## Cell 13 — Evaluation function

In [15]:
def evaluate_gnn(
    model,
    loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():

        for data in loader:

            data = data.to(device)

            output = model(data)

            probability = F.softmax(output, dim=1)
            prediction = output.argmax(dim=1)

            y_true.extend(
                data.y.cpu().numpy()
            )

            y_pred.extend(
                prediction.cpu().numpy()
            )

            y_prob.extend(
                probability.cpu().numpy()
            )

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    result = {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "F1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        )
    }

    try:
        result["ROC_AUC"] = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )
    except Exception:
        result["ROC_AUC"] = np.nan

    try:
        y_true_bin = label_binarize(
            y_true,
            classes=np.arange(num_classes)
        )

        result["PR_AUC"] = average_precision_score(
            y_true_bin,
            y_prob,
            average="macro"
        )

    except Exception:
        result["PR_AUC"] = np.nan

    return result

## Cell 14 — Training loop with early stopping

In [16]:
best_f1 = -np.inf
best_state = None

patience = 15
counter = 0

EPOCHS = 100

for epoch in range(1, EPOCHS + 1):

    loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_result = evaluate_gnn(
        model,
        val_loader
    )

    val_f1 = val_result["F1_macro"]

    print(
        f"Epoch {epoch:03d} | "
        f"Loss: {loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        counter = 0

    else:

        counter += 1

        if counter >= patience:
            print("Early stopping.")
            break

/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 001 | Loss: 1.9362 | Val F1: 0.1183


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 002 | Loss: 1.8615 | Val F1: 0.2079


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


...

Epoch 041 | Loss: 0.3721 | Val F1: 0.6738


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 042 | Loss: 0.3205 | Val F1: 0.6337


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 043 | Loss: 0.2778 | Val F1: 0.6487
Early stopping.


## Cell 15 — Load best model

In [17]:
model.load_state_dict(best_state)

model = model.to(device)

print("Best validation Macro-F1:", best_f1)

Best validation Macro-F1: 0.7064672031368046


## Cell 16 — Final test evaluation

In [18]:
start_time = time.time()

gnn_result = evaluate_gnn(
    model,
    test_loader
)

gnn_training_time = time.time() - start_time

gnn_result["Model"] = "Weighted GCN"
gnn_result["Training_Time_sec"] = gnn_training_time

print(gnn_result)

{'Accuracy': 0.7935483870967742, 'Precision_macro': 0.7038252671921832, 'Recall_macro': 0.6653867362563013, 'F1_macro': 0.6729121034245258, 'MCC': np.float64(0.675716142300657), 'ROC_AUC': np.float64(0.9348810050309166), 'PR_AUC': np.float64(0.7027576439944294), 'Model': 'Weighted GCN', 'Training_Time_sec': 0.13113141059875488}


## 17 Baseline Models 

In [21]:
LABELS_PATH = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL_labels.csv"  
IMAGE_DIR = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL"

assert os.path.exists(LABELS_PATH), f"labels csv : {LABELS_PATH}"
assert os.path.isdir(IMAGE_DIR), f"IMAGE_DIR : {IMAGE_DIR}"

labels_df = pd.read_csv("/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL_labels.csv")
print("Labels shape:", labels_df.shape)
display(labels_df.head())

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = nn.Identity()
resnet.eval()
resnet.to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def find_image_path(file_name, image_dir):
    for ext in ["", ".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
        candidate = os.path.join(image_dir, file_name + ext)
        if os.path.exists(candidate):
            return candidate
    for root, _, files in os.walk(image_dir):
        for ext in ["", ".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
            target = file_name + ext
            if target in files:
                return os.path.join(root, target)
    return None

feature_rows = []
missing_files = []

with torch.no_grad():
    for fname in tqdm(labels_df["file_name"], desc="Extracting ResNet features"):
        img_path = find_image_path(str(fname), IMAGE_DIR)
        if img_path is None:
            missing_files.append(fname)
            feature_rows.append(np.full(2048, np.nan))
            continue

        img = Image.open(img_path).convert("RGB")
        tensor = preprocess(img).unsqueeze(0).to(device)
        feat = resnet(tensor).cpu().numpy().flatten()
        feature_rows.append(feat)

feature_cols = [f"feat_{i}" for i in range(2048)]
features_df = pd.DataFrame(feature_rows, columns=feature_cols)

df = pd.concat(
    [labels_df.reset_index(drop=True), features_df],
    axis=1
)

if missing_files:
    print(f"\nWARNING: {len(missing_files)} images not found, dropping those rows.")
    df = df.dropna(subset=feature_cols).reset_index(drop=True)

print("Final df shape:", df.shape)
display(df.head())


Labels shape: (2064, 10)


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s] 
Extracting ResNet features: 100%|██████████| 2064/2064 [00:55<00:00, 36.97it/s]


Final df shape: (2064, 2058)


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight,...,feat_2038,feat_2039,feat_2040,feat_2041,feat_2042,feat_2043,feat_2044,feat_2045,feat_2046,feat_2047
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410,...,0.019225,0.000000,0.000944,0.395437,0.004008,0.073663,0.035710,0.00193,0.145052,0.300740
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265,...,0.000000,0.004709,0.069419,1.017439,0.000000,0.039197,1.756535,0.00000,0.030368,0.054500
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410,...,0.000000,0.000000,0.000000,0.091257,0.000000,0.021053,0.187012,0.00000,0.000000,0.000000
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321,...,0.040185,0.000000,0.003222,0.898902,0.000000,0.130383,0.302829,0.00000,0.028156,0.044233
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321,...,0.000000,0.000000,0.000000,1.287179,0.000000,0.201777,0.194737,0.00000,0.000000,0.042913


In [22]:
TARGET_COL = "disease"
GROUP_COL = "patient_id"

y = df[TARGET_COL].copy()
groups = df[GROUP_COL].copy()

non_feature_cols = [
    "file_name", "disease", "subcategory", "condition",
    "patient_id", "eye", "sex", "year",
    "image_width", "image_hight"
]
X = df.drop(columns=[c for c in non_feature_cols if c in df.columns])

print("X shape:", X.shape)
print("Classes:", y.nunique())
print("Class distribution:")
print(y.value_counts())


X shape: (2064, 2048)
Classes: 7
Class distribution:
disease
AMD    1231
NO      332
ERM     155
DME     147
RVO     101
VID      76
RAO      22
Name: count, dtype: int64


In [27]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print(
    "Patient overlap:",
    len(set(groups_train).intersection(set(groups_test)))
)

Train: (1665, 2048)
Test : (399, 2048)
Patient overlap: 0


In [28]:
gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx2, val_idx = next(
    gss_val.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

X_tr = X_train.iloc[train_idx2].copy()
X_val = X_train.iloc[val_idx].copy()

y_tr = y_train.iloc[train_idx2].copy()
y_val = y_train.iloc[val_idx].copy()

groups_tr = groups_train.iloc[train_idx2].copy()
groups_val = groups_train.iloc[val_idx].copy()

print("Train:", X_tr.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print(
    "Train-Val overlap:",
    len(set(groups_tr).intersection(set(groups_val)))
)


Train: (1349, 2048)
Validation: (316, 2048)
Test: (399, 2048)
Train-Val overlap: 0


In [29]:
X_tr = X_tr.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print("Missing values (train):", X_tr.isna().sum().sum())
print("Missing values (val)  :", X_val.isna().sum().sum())
print("Missing values (test) :", X_test.isna().sum().sum())

imputer = SimpleImputer(strategy="median")

X_tr_imp = pd.DataFrame(
    imputer.fit_transform(X_tr),
    columns=X_tr.columns,
    index=X_tr.index
)
X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Missing-value handling completed.")
print("X_tr_imp shape:", X_tr_imp.shape)


Missing values (train): 0
Missing values (val)  : 0
Missing values (test) : 0
Missing-value handling completed.
X_tr_imp shape: (1349, 2048)


In [30]:
scaler = StandardScaler()

X_tr_scaled = scaler.fit_transform(X_tr_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("Scaling completed.")


Scaling completed.


In [31]:
K = min(100, X_tr_scaled.shape[1])

selector = SelectKBest(
    score_func=f_classif,
    k=K
)

X_tr_sel = selector.fit_transform(X_tr_scaled, y_tr)
X_val_sel = selector.transform(X_val_scaled)
X_test_sel = selector.transform(X_test_scaled)

print("Selected features:", X_tr_sel.shape[1])


Selected features: 100


In [32]:
smote = SMOTE(
    random_state=SEED
)

X_tr_bal, y_tr_bal = smote.fit_resample(
    X_tr_sel,
    y_tr
)

print("Before SMOTE:")
print(y_tr.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_tr_bal).value_counts())

label_encoder = LabelEncoder()

y_tr_bal_enc = label_encoder.fit_transform(y_tr_bal)
y_test_enc = label_encoder.transform(y_test)

print("\nClasses:", list(label_encoder.classes_))
print("Encoded train label range:", y_tr_bal_enc.min(), "-", y_tr_bal_enc.max())

Before SMOTE:
disease
AMD    782
NO     228
ERM    100
DME     96
RVO     70
VID     56
RAO     17
Name: count, dtype: int64

After SMOTE:
disease
AMD    782
DME    782
ERM    782
NO     782
RAO    782
RVO    782
VID    782
Name: count, dtype: int64

Classes: ['AMD', 'DME', 'ERM', 'NO', 'RAO', 'RVO', 'VID']
Encoded train label range: 0 - 6


In [33]:
models_dict = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        n_jobs=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=SEED
    ),

    "k-NN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=SEED
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        num_leaves=31,
        random_state=SEED,
        verbosity=-1
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=300,
        early_stopping=True,
        random_state=SEED
    )
}

In [34]:
def evaluate_model(model, X_train, y_train, X_test, y_test, training_time):

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
    else:
        y_prob = None

    result = {
        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision_macro": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall_macro": recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "F1_macro": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y_test,
            y_pred
        ),

        "Training_Time_sec": training_time
    }

    if y_prob is not None:
        try:
            result["ROC_AUC"] = roc_auc_score(
                y_test,
                y_prob,
                multi_class="ovr",
                average="macro"
            )

            y_test_bin = label_binarize(
                y_test,
                classes=np.unique(y_test)
            )

            result["PR_AUC"] = average_precision_score(
                y_test_bin,
                y_prob,
                average="macro"
            )

        except Exception:
            result["ROC_AUC"] = np.nan
            result["PR_AUC"] = np.nan

    else:
        result["ROC_AUC"] = np.nan
        result["PR_AUC"] = np.nan

    return result


In [35]:
baseline_results = []
trained_models = {}

for name, model in models_dict.items():

    print(f"\nTraining: {name}")

    start_time = time.time()
    model.fit(X_tr_bal, y_tr_bal_enc)
    training_time = time.time() - start_time

    result = evaluate_model(
        model,
        X_tr_bal,
        y_tr_bal_enc,
        X_test_sel,
        y_test_enc,
        training_time
    )

    result["Model"] = name

    baseline_results.append(result)
    trained_models[name] = model

    print(result)


Training: Logistic Regression
{'Accuracy': 0.7092731829573935, 'Precision_macro': 0.48271420390606096, 'Recall_macro': 0.6101805825836274, 'F1_macro': 0.5264099385516527, 'MCC': np.float64(0.5611185170946148), 'Training_Time_sec': 1.8080790042877197, 'ROC_AUC': np.float64(0.9005390467225177), 'PR_AUC': np.float64(0.5460983907540637), 'Model': 'Logistic Regression'}

Training: SVM
{'Accuracy': 0.7844611528822055, 'Precision_macro': 0.6249016296194518, 'Recall_macro': 0.6075647522372459, 'F1_macro': 0.6048593929894298, 'MCC': np.float64(0.628356072640684), 'Training_Time_sec': 4.334466457366943, 'ROC_AUC': np.float64(0.9151283064654155), 'PR_AUC': np.float64(0.6199353841737368), 'Model': 'SVM'}

Training: k-NN
{'Accuracy': 0.506265664160401, 'Precision_macro': 0.3878435758624446, 'Recall_macro': 0.5717499718648271, 'F1_macro': 0.41388582232267535, 'MCC': np.float64(0.4008808767550582), 'Training_Time_sec': 0.0011913776397705078, 'ROC_AUC': np.float64(0.7845971359510518), 'PR_AUC': np.fl

In [36]:
baseline_df = pd.DataFrame(baseline_results)

baseline_df = baseline_df[
    [
        "Model",
        "Accuracy",
        "Precision_macro",
        "Recall_macro",
        "F1_macro",
        "ROC_AUC",
        "PR_AUC",
        "MCC",
        "Training_Time_sec"
    ]
].sort_values(
    "F1_macro",
    ascending=False
).reset_index(drop=True)

display(baseline_df)


,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,ROC_AUC,PR_AUC,MCC,Training_Time_sec
0,SVM,0.784461,0.624902,0.607565,0.604859,0.915128,0.619935,0.628356,4.334466
1,XGBoost,0.774436,0.618677,0.604116,0.598095,0.880718,0.621688,0.613484,6.797480
2,Gradient Boosting,0.764411,0.598256,0.592293,0.586564,0.856698,0.594225,0.597888,186.206362
3,MLP,0.761905,0.579087,0.607910,0.579351,0.880349,0.586459,0.601579,1.180280
4,LightGBM,0.766917,0.655766,0.574185,0.577163,0.886135,0.619497,0.595752,4.926400
5,Random Forest,0.726817,0.698325,0.505975,0.548738,0.858815,0.574008,0.505907,4.358570
6,Logistic Regression,0.709273,0.482714,0.610181,0.526410,0.900539,0.546098,0.561119,1.808079
7,k-NN,0.506266,0.387844,0.571750,0.413886,0.784597,0.362916,0.400881,0.001191
8,Decision Tree,0.563910,0.340658,0.411299,0.363989,0.660559,0.232099,0.325393,0.940925


## Cell 18 — Compare GNN with baseline models



In [37]:

comparison = baseline_df.copy()

gnn_row = pd.DataFrame([{
    "Model": "Proposed Weighted GCN",
    "Accuracy": gnn_result["Accuracy"],
    "Precision_macro": gnn_result["Precision_macro"],
    "Recall_macro": gnn_result["Recall_macro"],
    "F1_macro": gnn_result["F1_macro"],
    "ROC_AUC": gnn_result["ROC_AUC"],
    "PR_AUC": gnn_result["PR_AUC"],
    "MCC": gnn_result["MCC"],
    "Training_Time_sec": gnn_result["Training_Time_sec"]
}])

comparison = pd.concat(
    [comparison, gnn_row],
    ignore_index=True
)

comparison = comparison.sort_values(
    "F1_macro",
    ascending=False
).reset_index(drop=True)

display(comparison)

,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,ROC_AUC,PR_AUC,MCC,Training_Time_sec
0,Proposed Weighted GCN,0.793548,0.703825,0.665387,0.672912,0.934881,0.702758,0.675716,0.131131
1,SVM,0.784461,0.624902,0.607565,0.604859,0.915128,0.619935,0.628356,4.334466
2,XGBoost,0.774436,0.618677,0.604116,0.598095,0.880718,0.621688,0.613484,6.797480
3,Gradient Boosting,0.764411,0.598256,0.592293,0.586564,0.856698,0.594225,0.597888,186.206362
4,MLP,0.761905,0.579087,0.607910,0.579351,0.880349,0.586459,0.601579,1.180280
5,LightGBM,0.766917,0.655766,0.574185,0.577163,0.886135,0.619497,0.595752,4.926400
6,Random Forest,0.726817,0.698325,0.505975,0.548738,0.858815,0.574008,0.505907,4.358570
7,Logistic Regression,0.709273,0.482714,0.610181,0.526410,0.900539,0.546098,0.561119,1.808079
8,k-NN,0.506266,0.387844,0.571750,0.413886,0.784597,0.362916,0.400881,0.001191
9,Decision Tree,0.563910,0.340658,0.411299,0.363989,0.660559,0.232099,0.325393,0.940925
